# Amostragem em PINNs

Este *notebook* tem como objetivo explorar diferentes tipos de amostragem, buscando mostrar como isso influencia na previsão de uma PINN.

**Autor:** Edélio Gabriel.

## A magia da amostragem

### **Um pouco de estatística**

<div style="text-align: center;">
  <img 
    src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*1CWoFdI5DeY13tSohGL_Nw.jpeg" 
    alt="Ilustração do conceito de aomstragem"
    style="max-width: 400px; width: 100%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://medium.com/@rafaelporfirio/m%C3%A9todos-de-amostragem-na-estat%C3%ADstica-375e266b6150' target='_blank'>
    Medium - Métodos de Amostragem na Estatística
  </a>
</div>

Existem alguns conceitos importantes no campo da Estatística. O primeiro deles é, na verdade, um dos ramos da Estatística: a **Estatítica Descritiva**. Ela se ocupa em buscar organizar, resumir e descrever conjuntos de dados, transformando informação em métricas como média, mediana, moda e desvio padrão. 

Ora, mas nem sempre conseguimos ter acesso a todos os dados, o que dificulta a tarefa de **Inferência**: chegar a concllusões ou deduzir resultados a partir de dados.

Ora, mas nem sempre temos acesso a todos os dados. Em outras palavras, nem sempre conseguimos os dados da **população**, o grupo completo sobre o qual desejamos fazer nossa análise. Por isso, é rotineiro trabalharmos com **amostras**, uma subcoleção de indivíduos constituintes da população.

O que buscamos é conseguir representar o todo a partir de suas partes! Por isso, é sempre importante que nossas amostras sejam representativas: incorporem todas - ou pelo menos as principais - características da nossa população de análise.

### **Os tipos de amostragem**

É pensando no que foi discutido acima que foram desenvolvidos deferentes tipos de amostragem, buscando recuperar as características de uma população (de dados) a partir das suas amostras.

Os tipos de amostragem podem ser divididos em duas grandes categorias:

``1. Amostragem Probabilística``

Nesse tipo de amostragem, todos os elementos da população possuem uma probabilidade conhecida de serem selecionados.

Tipos:
- **Amostragem aleatória simples**
- **Amostragem aleatória sistemática**
- **Amostragem estratificada**
- **Amostragem por conglomerados**

Características:
- Seleção aleatória dos participantes;
- Maior representatividade da população;
- Permite generalizações estatísticas;
- Menor viés de seleção.

``2. Amostragem Não-Probabilística``

Nesse tipo de amostragem, os participantes são selecionados por critérios não aleatórios.

Tipos:
- **Amostragem por conveniência**
- **Amostragem por auto-seleção**
- **Amostragem intencional**
- **Amostragem por bola de neve**
- **Amostragem por quotas**

Características:
- Mais simples e barata;
- Coleta geralmente mais rápida;
- Menor capacidade de generalização;
- Maior suscetibilidade a vieses.

## Como a amostragem é usada no contexto de PINNs?

Como você deve ter visto nos *notebooks* de exemplo, o processo de amostragem de dados é parte integrante da implementação de PINNs. Isso é direto quando entendemos que essa abordagem foi desenvolvida justamente para casos em que a obtenção de dados é difícil, esparsa ou não garante representatividade completa do problema.

O paper original de Raissi et al. [[ref]](#original-paper) não explora esse tópico com profundidade, concentrando-se nas aplicações. Os artigos de revisão mais abrangentes, como Karniadakis et al. [[ref]](#review-paper) e Cuomo et al. [[ref]](#cuomo-paper), mencionam a amostragem como um fator relevante, mas sem uma análise sistemática. É na literatura mais recente e especializada que o tema ganha atenção — trabalhos como Lu et al. [[ref]](#deepxde-paper), que introduz a biblioteca DeepXDE com estratégias de amostragem adaptativa, e Nabian et al. [[ref]](#nabian-paper), que analisa o impacto de diferentes distribuições de pontos de colocação na convergência.

O que exploraremos aqui é justamente isso: **como a escolha da estratégia de amostragem influencia a previsão da PINN** — tanto em termos de precisão quanto de velocidade de convergência. As principais estratégias que consideraremos são:

- **Amostragem uniforme** (*uniform sampling*) — pontos distribuídos regularmente no domínio
- **Amostragem aleatória** (*random sampling*) — pontos sorteados aleatoriamente
- **Latin Hypercube Sampling (LHS)** — estratégia quasi-aleatória que garante melhor cobertura do domínio [[ref]](#lhs-paper)
- **Amostragem adaptativa** (*adaptive sampling*) — pontos concentrados nas regiões onde o resíduo é maior, refinando progressivamente a solução [[ref]](#lu-adaptive)

## Nosso problema

Como discutido anteriormente, muitos problemas físicos podem ser descritos por Equações Diferenciais Parciais (EDPs) associadas a estados de equilíbrio. Nesse exemplo, inspirado em uma das aplicações do artigo de Baty et. al. [[ref]](#hands-on-paper), trabalharemos com a **equação de Helmholtz**, uma EDP utilizada em problemas envolvendo ondas, eletromagnetismo, acústica e estruturas magnéticas em plasmas astrofísicos.

---

### `Requisitos teóricos`

#### **Contexto físico**

A equação de Helmholtz pode descrever configurações magnéticas estacionárias presentes na coroa solar.

Um exemplo importante são os chamados **arcades magnéticos solares**, estruturas em forma de arco produzidas pelas linhas de campo magnético emergindo da superfície do Sol.

<div style="text-align: center;">
  <img 
    src="https://s2.glbimg.com/NjvZ20nQXHVlRlA-lRJqiXM7hU8=/e.glbimg.com/og/ed/f/original/2016/03/17/campos_magneticos_sol_aia171_sdo_070114.jpg"
    alt="Arcades magnéticos solares"
    style="max-width: 500px; width: 60%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://commons.wikimedia.org/wiki/File:Solar_coronal_loops.jpg' target='_blank'>
    Revista Galileu — NASA divulga mapa magnético do Sol
  </a>
</div>

Essas estruturas podem ser modeladas assumindo equilíbrio magnetostático e simetria translacional em uma das direções espaciais.

Nesse cenário, introduz-se uma função escalar:

$$
u(x,z)
$$

conhecida como **função de fluxo magnético** (*magnetic flux function*).

As curvas de nível de $u$ representam as linhas de campo magnético no plano $(x,z)$.

---

#### **Equação de Helmholtz**

A dinâmica espacial da função de fluxo é modelada pela equação:

$$
\nabla^2 u + c^2 u = 0
\tag{1}
$$

onde:

- $u(x,z)$ é a função de fluxo magnético;
- $c$ é uma constante;
- $\nabla^2$ é o operador Laplaciano cartesiano.

Em duas dimensões:

$$
\nabla^2
=
\frac{\partial^2}{\partial x^2}
+
\frac{\partial^2}{\partial z^2}
\tag{2}
$$

O domínio espacial considerado é:

$$
(x,z)\in\left[-\frac{L}{2},\frac{L}{2}\right]
$$

Fisicamente:

- $x$ representa a direção horizontal;
- $z$ representa a altitude acima da superfície solar;
- a superfície do Sol encontra-se em:

$$
z=0
$$

---

#### **Campo magnético associado**

A partir da função de fluxo $u(x,z)$, é possível reconstruir o campo magnético total.

Assumindo simetria translacional na direção $y$, o campo magnético pode ser escrito como:

$$
\mathbf{B}(x,z)
=
\nabla u(x,z)\times \mathbf{e}_y
+
B_y(u)\mathbf{e}_y
\tag{3}
$$

onde:

- $\mathbf{e}_y$ é o vetor unitário na direção $y$;
- $B_y$ representa a componente do campo magnético nessa direção.

A primeira parcela descreve as componentes do campo no plano $(x,z)$, enquanto a segunda adiciona uma componente longitudinal.

As linhas de campo magnético podem então ser obtidas diretamente pelas curvas de nível da função $u$.

---

#### **Solução analítica via séries de Fourier**

Uma solução analítica para estruturas do tipo *triple arcade* pode ser construída utilizando séries de Fourier:

$$
u(x,z)
=
\sum_{k=1}^{3}
\exp(-\nu z)
\left[
a_k
\cos\left(
\frac{k\pi}{L}x
\right)
\right]
\tag{4}
$$

onde:

- $a_k$ são coeficientes associados aos diferentes modos espaciais;
- $\nu$ controla o decaimento vertical da solução;
- os termos cossenoidais descrevem a periodicidade em $x$.

Essa solução representa uma superposição de modos magnéticos periódicos ao longo da superfície solar.

Fisicamente:

- os termos exponenciais modelam o enfraquecimento do campo com a altitude;
- os modos de Fourier controlam a geometria dos arcades magnéticos.

---

#### **Relação de dispersão**

Substituindo a solução anterior na equação de Helmholtz, obtemos a relação:

$$
\nu^2
=
\frac{k^2\pi^2}{L^2}
-
c^2
\tag{5}
$$

Essa expressão conecta:

- a frequência espacial do modo;
- a taxa de decaimento vertical;
- o parâmetro físico $c$ da equação.

Observe que diferentes valores de $k$ produzem diferentes estruturas espaciais no campo magnético.

## Aplicando a PINN

O código completo está localizado na pasta `scripts`, especificamente no arquivo `ex04_pinn_inverse_stationary.py`. Para facilitar a discussão, colocarei apenas trechos necessários para uma compreensão mais aprofundada.

---

A célula seguinte serve para:

- Recarregar automaticamente qualquer arquivo que for editado nos scripts
- Encontrar a pasta dos *scripts*, permitindo importar as funções criadas

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath("../scripts"))

import plotly.io as pio
pio.renderers.default = "notebook"

### **Importações necessárias**

In [2]:
import torch.nn as nn
import torch
import numpy as np
from geral_functions import PINN
from sampling_helmholtz import (
    sample_uniform, sample_random, sample_lhs,
    sample_boundary_helmholtz,
    train_helmholtz, evaluate_helmholtz
)

from plot_utils import (
    plot_l2_comparison,
    plot_heatmaps_comparison,
    plot_loss_comparison,
    plot_sampling_points)

### **Parâmetros do problema**

Os hiperparâmetros utilizados na arquitetura da rede neural e no processo de treinamento foram inspirados na Os valores dos parâmetros que envolvem a arquitetura da rede a amostragem foram inspirados nos usados por Baty em "***A hands-on introduction to physics-informed neural networks for solving partial differential equations with benchmark tests taken from astrophysics and plasma physics.***"[[ref]](#hands-on-paper).

In [3]:
# Definição do local onde o código serpa executado. Por padrão, gpu
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {DEVICE}')

# Arquitetura da rede
N_INPUTS   = 2
N_OUTPUTS  = 1
N_HIDDEN   = 16
N_LAYERS   = 7
ACTIVATION = nn.Tanh

# Parâmetros de amostragem
N_COLLOC   = 200
N_BC       = 50
N_EPOCHS   = 10000

# Parâmetros para o treinamento
LR         = 1e-4

Usando: cuda


### **Amostragem dos pontos**

In [4]:
X_BC, U_BC = sample_boundary_helmholtz(N_BC, DEVICE)

In [5]:
point_sets = [
    sample_uniform(N_COLLOC, DEVICE).detach().cpu().numpy(),
    sample_random(N_COLLOC, DEVICE).detach().cpu().numpy(),
    sample_lhs(N_COLLOC, DEVICE).detach().cpu().numpy(),
]
plot_sampling_points(point_sets, ['Uniforme', 'Aleatória', 'LHS'])

### **Treinamento**

In [6]:
results = {}

N_RUNS = 10
SEEDS  = range(N_RUNS)

for strategy, sample_fn in [
    ('Uniforme', sample_uniform),
    ('Aleatória', sample_random),
    ('LHS', sample_lhs),
]:

    runs = []

    print(f'\n===== {strategy} =====')

    for seed in SEEDS:

        # reinicializa pesos da rede
        torch.manual_seed(seed)

        model = PINN(
            N_INPUTS,
            N_OUTPUTS,
            N_HIDDEN,
            N_LAYERS,
            ACTIVATION
        ).to(DEVICE)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LR
        )

        # amostragem
        X_COL = sample_fn(
            N_COLLOC,
            DEVICE,
            seed=seed
        )

        # treinamento
        history = train_helmholtz(
            model,
            optimizer,
            X_COL,
            X_BC,
            U_BC,
            N_EPOCHS
        )

        # avaliação
        metrics = evaluate_helmholtz(model, DEVICE)
        metrics['history'] = history
        metrics['seed'] = seed

        runs.append(metrics)

        print(
            f'Run {seed:02d} | '
            f'L2 Error = {metrics["l2_error"]:.2e}'
        )

    # -----------------------------
    # estatísticas agregadas
    # -----------------------------

    l2_errors = [r['l2_error'] for r in runs]

    results[strategy] = {

        # todas execuções
        'runs': runs,

        # estatísticas
        'l2_mean': np.mean(l2_errors),
        'l2_std':  np.std(l2_errors),

        # compatibilidade com plots antigos
        **runs[-1]
    }

    print(
        f'\n{strategy}'
        f'\nL2 médio = {results[strategy]["l2_mean"]:.2e}'
        f'\nDesvio   = {results[strategy]["l2_std"]:.2e}\n'
    )


===== Uniforme =====


/home/edelio25024/miniconda3/envs/ilumpy/lib/python3.14/site-packages/torch/autograd/graph.py:841: UserWarning:

Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)



Epoch 00000 | Loss: 2.19e-01 | Loss PDE: 1.70e-03 | Loss BC: 2.18e-01


Epoch 01000 | Loss: 9.40e-02 | Loss PDE: 8.99e-03 | Loss BC: 8.50e-02


Epoch 02000 | Loss: 2.44e-02 | Loss PDE: 5.16e-03 | Loss BC: 1.93e-02


...

Epoch 06000 | Loss: 4.18e-03 | Loss PDE: 6.25e-04 | Loss BC: 3.55e-03


Epoch 07000 | Loss: 2.17e-03 | Loss PDE: 4.83e-04 | Loss BC: 1.68e-03


Epoch 08000 | Loss: 1.01e-03 | Loss PDE: 2.67e-04 | Loss BC: 7.46e-04


Epoch 09000 | Loss: 4.58e-04 | Loss PDE: 1.53e-04 | Loss BC: 3.04e-04


Run 09 | L2 Error = 1.39e-02

LHS
L2 médio = 2.41e-02
Desvio   = 1.03e-02



In [7]:
histories     = [results[s]['history'] for s in ['Uniforme', 'Aleatória', 'LHS']]
results_list  = [results[s] for s in ['Uniforme', 'Aleatória', 'LHS']]
labels        = ['Uniforme', 'Aleatória', 'LHS']

plot_loss_comparison(histories, labels)
plot_heatmaps_comparison(results_list, labels)
plot_l2_comparison(results_list, labels)

## Referências

<a id="original-paper"></a> [RAISSI, Maziar; PERDIKARIS, Paris; KARNIADAKIS, George Em. Physics informed deep learning (part i): Data-driven solutions of nonlinear partial differential equations. arXiv preprint arXiv:1711.10561, 2017.](https://arxiv.org/abs/1711.10561)

<a id="review-paper"></a> [KARNIADAKIS, George Em et al. Physics-informed machine learning. Nature Reviews Physics, v. 3, p. 422–440, 2021.](https://www.nature.com/articles/s42254-021-00314-5)

<a id="cuomo-paper"></a> [CUOMO, Salvatore et al. Scientific machine learning through physics-informed neural networks: Where we are and what's next. Journal of Scientific Computing, v. 92, n. 88, 2022.](https://link.springer.com/article/10.1007/s10915-022-01939-z)

<a id="deepxde-paper"></a> [LU, Lu et al. DeepXDE: A deep learning library for solving differential equations. SIAM Review, v. 63, n. 1, p. 208–228, 2021.](https://epubs.siam.org/doi/10.1137/19M1274067)

<a id="nabian-paper"></a> [NABIAN, Mohammad Amin; GLADSTONE, Rini Jasmine; MEIDANI, Hadi. Efficient training of physics-informed neural networks via importance sampling. Computer-Aided Civil and Infrastructure Engineering, v. 36, n. 8, p. 962–977, 2021.](https://onlinelibrary.wiley.com/doi/10.1111/mice.12685)

<a id="lhs-paper"></a> [MCKAY, Michael D.; BECKMAN, Richard J.; CONOVER, William J. A comparison of three methods for selecting values of input variables in the analysis of output from a computer code. Technometrics, v. 21, n. 2, p. 239–245, 1979.](https://www.tandfonline.com/doi/abs/10.1080/00401706.1979.10489755)

<a id="lu-adaptive"></a> [LU, Lu et al. Adaptive training strategies for physics-informed neural networks. arXiv preprint arXiv:2211.15997, 2022.](https://arxiv.org/abs/2211.15997)

<a id="hands-on-paper"> </a> [BATY, Hubert. A hands-on introduction to physics-informed neural networks for solving partial differential equations with benchmark tests taken from astrophysics and plasma physics. arXiv preprint arXiv:2403.00599, 2024.](https://arxiv.org/html/2403.00599v1#S2)

#